# Satellite close-approach hackathon

You get a **catalog of ~17k satellites** (orbit data from **1 June 2026**). Write an algorithm that finds **close approaches**: pairs of satellites that come within a distance threshold while you propagate their orbits forward in time.

### Goal
Implement `find_close_approaches` in the solution cell. Return claims that the verifier accepts. Beat the naive baseline.

### Winning rule
Find as many **verified True** claims as possible in **1 minute**. If tied, the **faster** run wins.

### Inputs you receive (both, already prepared)
| Argument | Type | Role |
|----------|------|------|
| `catalog` | `Catalog` | Full orbital **data** (inclination, eccentricity, names, …) for filtering / screening |
| `satellites` | `Dict[int, EarthSatellite]` | Same objects as **SGP4 propagators** — ready for `propagate_many` |

`satellites` is built **once before the timer** so your 1 minute is not spent on `catalog_to_satellites`.

## 0. Setup (Google Colab)

Run the next cell once (or **Runtime → Run all**). It installs packages and downloads the catalog + toolkit from the `solve` branch zip.

In [ ]:
import os
import sys
import shutil
import subprocess
import time
import zipfile
from pathlib import Path
from urllib.request import Request, urlopen

pkgs = [
    "skyfield>=1.48",
    "sgp4>=2.23",
    "numpy>=1.26",
    "scipy>=1.11",
    "plotly>=5.18",
    "pandas>=2.1",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

CONTENT = Path("/content") if Path("/content").exists() else Path.cwd()
ZIP_PATH = CONTENT / "solve_branch.zip"
ROOT = CONTENT / "hackathon"
ZIP_URLS = [
    f"https://codeload.github.com/abensour/collision_detection_hackathon/zip/refs/heads/solve?t={int(time.time())}",
    f"https://github.com/abensour/collision_detection_hackathon/archive/refs/heads/solve.zip?t={int(time.time())}",
]


def _download(url: str, dest: Path) -> None:
    print("Downloading", url.split("?")[0], "…")
    req = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(req, timeout=180) as resp, dest.open("wb") as out:
        shutil.copyfileobj(resp, out)


last_error = None
for url in ZIP_URLS:
    try:
        _download(url, ZIP_PATH)
        last_error = None
        break
    except Exception as exc:
        last_error = exc
        print("Download failed:", exc)
if last_error is not None:
    raise RuntimeError("Could not download the student files from GitHub") from last_error

zip_bytes = ZIP_PATH.stat().st_size
print(f"Downloaded {zip_bytes / 1e6:.2f} MB")
if zip_bytes < 500_000:
    raise RuntimeError(f"Zip is too small ({zip_bytes} bytes) — download did not get the catalog")

extract_parent = CONTENT / "_hackathon_extract"
if extract_parent.exists():
    shutil.rmtree(extract_parent)
extract_parent.mkdir(parents=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(extract_parent)

inner_dirs = [p for p in extract_parent.iterdir() if p.is_dir()]
inner = inner_dirs[0] if len(inner_dirs) == 1 else extract_parent

if ROOT.exists():
    shutil.rmtree(ROOT)
shutil.copytree(inner, ROOT)
shutil.rmtree(extract_parent, ignore_errors=True)

required = [
    ROOT / "spacetrack_data.json",
    ROOT / "conjunction_toolkit" / "__init__.py",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing from zip: " + ", ".join(missing))

catalog_bytes = (ROOT / "spacetrack_data.json").stat().st_size
print(f"Catalog spacetrack_data.json: {catalog_bytes / 1e6:.2f} MB")
if catalog_bytes < 1_000_000:
    raise RuntimeError("Catalog file is too small — data did not download")

print("Files in /content/hackathon:")
for path in sorted(ROOT.rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts and path.suffix != ".ipynb":
        print(" ", path.relative_to(ROOT))

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Ready. Project root:", ROOT)
print("Implement your algorithm in the 'Your solution' cell below.")

## 1. Imports and shared helpers

This cell imports the toolkit functions you may call from your solution.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timedelta
from itertools import combinations
from typing import Callable, Dict, List, Tuple
import time

import numpy as np
from skyfield.api import EarthSatellite

from conjunction_toolkit import (
    Catalog,
    ConjunctionClaim,
    OrbitalElements,
    catalog_to_satellites,
    closest_approach_on_grid,
    load_default_catalog,
    plot_pair_with_distance,
    plot_trajectories,
    propagate_many,
    save_html,
    time_grid,
)
from conjunction_toolkit.propagate import datetimes_of

OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Earth radius used by SGP4 (WGS72), in km — needed only to convert SGP4 altitude units
WGS72_EARTH_RADIUS_KM = 6378.135

## 2. Load the catalog

`catalog: Catalog` holds one `OrbitalElements` row per object (id, name, epoch, inclination, …).

Later we also build `satellites: Dict[int, EarthSatellite]` once (same objects, but as SGP4 propagators). Your solution gets **both**.

In [ ]:
catalog: Catalog = load_default_catalog()

print(f"Source  : {catalog.source_path}")
print(f"Objects : {len(catalog)}")

epochs: List[datetime] = [obj.epoch_utc for obj in catalog]
print(f"Epochs  : {min(epochs).date()} → {max(epochs).date()}")

print("\nSample objects (OrbitalElements fields):")
for object_id in catalog.ids()[:5]:
    obj: OrbitalElements = catalog[object_id]
    print(
        f"  id={obj.norad_cat_id:>6}  {obj.object_name:<28}  "
        f"inc={obj.inclination_deg:6.2f}°  e={obj.eccentricity:.4f}  "
        f"n={obj.mean_motion_rev_per_day:.4f} rev/day  epoch={obj.epoch_utc.date()}"
    )

## 3. Playground — explore the catalog (before you code)

This section is only for understanding the data. Nothing here is graded.

### Why look at these fields?

| Field | Type | Why it matters for close approaches |
|-------|------|-------------------------------------|
| `inclination_deg` | `float` (degrees) | Orbit tilt vs the equator. Satellites on very different inclinations often never get near each other. |
| Perigee / apogee altitude | `float` (km above Earth) | How high the satellite flies. Objects in very different altitude bands are unlikely to meet. |

### Where the numbers come from

- **Inclination** is already on each catalog row (`OrbitalElements.inclination_deg`).
- **Altitude min/max** come from SGP4 after `catalog_to_satellites(...)`:
  - `sat.model.altp` = perigee altitude (in Earth radii)
  - `sat.model.alta` = apogee altitude (in Earth radii)
  - multiply by `WGS72_EARTH_RADIUS_KM` to get kilometers

We do **not** invent our own altitude formula — we use the same SGP4 model the toolkit uses to propagate.

In [ ]:
# --- types from the catalog / toolkit ---
# catalog[id] -> OrbitalElements  (id, name, epoch, inclination_deg, eccentricity, ...)
# catalog_to_satellites(...) -> Dict[int, EarthSatellite]  (ready to propagate with SGP4)

def sgp4_altitude_range_km(sat: EarthSatellite) -> Tuple[float, float]:
    """Return (perigee_km, apogee_km) using SGP4 fields, not a hand-written orbit formula.

    sat.model.altp / alta are altitudes in Earth radii (WGS72).
    """
    perigee_km = float(sat.model.altp) * WGS72_EARTH_RADIUS_KM
    apogee_km = float(sat.model.alta) * WGS72_EARTH_RADIUS_KM
    return perigee_km, apogee_km


# 1) Inclination — available directly on every catalog row (no propagation needed)
objects: List[OrbitalElements] = list(catalog)
inclinations_deg: np.ndarray = np.array(
    [obj.inclination_deg for obj in objects],
    dtype=float,
)

print("=== Inclination (degrees) ===")
print("Reason: pairs with very different tilts rarely meet.")
print(
    f"  count={len(inclinations_deg)}  "
    f"min={inclinations_deg.min():.2f}  max={inclinations_deg.max():.2f}  "
    f"mean={inclinations_deg.mean():.2f}  median={np.median(inclinations_deg):.2f}"
)
print("  bins:")
bins = [0, 30, 60, 90, 120, 180]
hist, _ = np.histogram(inclinations_deg, bins=bins)
for i, count in enumerate(hist):
    print(f"    {bins[i]:>3}–{bins[i + 1]:<3}° : {count}")


# 2) Altitude range — use SGP4 on a sample (building all ~17k satellites is slow for a playground)
sample_ids: List[int] = catalog.ids()[::20]  # every 20th object ≈ 5% sample
sample_sats: Dict[int, EarthSatellite] = catalog_to_satellites(catalog, norad_ids=sample_ids)

perigee_km: List[float] = []
apogee_km: List[float] = []
for sat in sample_sats.values():
    lo, hi = sgp4_altitude_range_km(sat)
    perigee_km.append(lo)
    apogee_km.append(hi)

perigee_arr = np.array(perigee_km, dtype=float)
apogee_arr = np.array(apogee_km, dtype=float)

print("\n=== Altitude above Earth (km), from SGP4 altp/alta ===")
print("Reason: objects in very different height bands rarely meet.")
print(f"  sample size: {len(sample_sats)} satellites (subset for speed)")
print(
    f"  perigee (lowest point): min={perigee_arr.min():.1f}  "
    f"max={perigee_arr.max():.1f}  mean={perigee_arr.mean():.1f}"
)
print(
    f"  apogee  (highest point): min={apogee_arr.min():.1f}  "
    f"max={apogee_arr.max():.1f}  mean={apogee_arr.mean():.1f}"
)


# 3) A few well-known satellites — full typed view of one catalog row + SGP4 heights
print("\n=== Example satellites (catalog fields + SGP4 heights) ===")
for name in ("ISS (ZARYA)", "HST", "CSS (TIANHE)"):
    matches: List[OrbitalElements] = list(catalog.filter_by_name(name))
    if not matches:
        continue
    obj: OrbitalElements = sorted(matches, key=lambda o: o.epoch_utc, reverse=True)[0]
    sat: EarthSatellite = catalog_to_satellites(catalog, norad_ids=[obj.norad_cat_id])[obj.norad_cat_id]
    lo, hi = sgp4_altitude_range_km(sat)
    print(f"  name         : {obj.object_name}")
    print(f"  norad id     : {obj.norad_cat_id}")
    print(f"  epoch (UTC)  : {obj.epoch_utc.isoformat()}")
    print(f"  inclination  : {obj.inclination_deg:.2f} deg")
    print(f"  eccentricity : {obj.eccentricity:.6f}")
    print(f"  height band  : {lo:.1f} – {hi:.1f} km  (SGP4 perigee–apogee)")
    print()

## 4. Propagate and plot a few orbits

Quick visual check that propagation works. The Earth sphere in the plot does not rotate.

In [ ]:
demo_ids: List[int] = []
for name in ("ISS (ZARYA)", "HST", "CSS (TIANHE)"):
    matches: List[OrbitalElements] = list(catalog.filter_by_name(name))
    if matches:
        matches.sort(key=lambda obj: obj.epoch_utc, reverse=True)
        demo_ids.append(matches[0].norad_cat_id)

if len(demo_ids) < 2:
    recent: List[OrbitalElements] = sorted(
        catalog, key=lambda obj: obj.epoch_utc, reverse=True
    )
    demo_ids = [obj.norad_cat_id for obj in recent[:3]]

demo_ids = demo_ids[:3]
demo_satellites: Dict[int, EarthSatellite] = catalog_to_satellites(
    catalog, norad_ids=demo_ids
)

demo_from: datetime = max(catalog[i].epoch_utc for i in demo_ids)
demo_until: datetime = demo_from + timedelta(hours=6)
orbit_times = time_grid(demo_from, demo_until, step_seconds=60)

print("Objects:", [(i, demo_satellites[i].name) for i in demo_ids])
print(f"Propagate from : {demo_from.isoformat()}")
print(f"Propagate until: {demo_until.isoformat()}")

fig = plot_trajectories(demo_satellites, orbit_times, title="Sample orbits")
save_html(fig, OUTPUT_DIR / "notebook_trajectories.html")
fig.show()

## 5. Shared evaluation settings

Used by the naive baseline, your solution, and the verifier.

| Setting | Value | Meaning |
|---------|-------|---------|
| `PROPAGATE_FOR_HOURS` | 6 | How far ahead to search |
| `CLOSE_APPROACH_THRESHOLD_KM` | **10** | True if distance at claimed time ≤ 10 km |
| `MAX_RUNTIME_SECONDS` | **60** | Official 1-minute limit for your solution |
| `NAIVE_RUNTIME_SECONDS` | 30 | Naive demo stops earlier |

This cell also builds `satellites = catalog_to_satellites(catalog)` **once**. That conversion is **outside** the timed run.

In [ ]:
PROPAGATE_FOR_HOURS: int = 6
CLOSE_APPROACH_THRESHOLD_KM: float = 10.0
MAX_RUNTIME_SECONDS: float = 60.0       # 1-minute official limit
NAIVE_RUNTIME_SECONDS: float = 30.0     # naive demo stop

epochs: List[datetime] = sorted(obj.epoch_utc for obj in catalog)
propagate_from_utc: datetime = epochs[len(epochs) // 2]
propagate_until_utc: datetime = propagate_from_utc + timedelta(hours=PROPAGATE_FOR_HOURS)

# Built once here — NOT counted inside find_close_approaches runtime
print("Building SGP4 satellites from catalog (one-time, outside the timer)…")
t_build = time.perf_counter()
satellites: Dict[int, EarthSatellite] = catalog_to_satellites(catalog)
print(f"Built {len(satellites)} satellites in {time.perf_counter() - t_build:.2f} s")

n_objects: int = len(catalog)
n_pairs: int = n_objects * (n_objects - 1) // 2

print(f"Catalog objects : {n_objects:,}")
print(f"Unique pairs    : {n_pairs:,}")
print(f"Propagate from  : {propagate_from_utc.isoformat()}")
print(f"Propagate until : {propagate_until_utc.isoformat()}  ({PROPAGATE_FOR_HOURS} h)")
print(f"Threshold       : {CLOSE_APPROACH_THRESHOLD_KM} km")
print(f"Your time limit : {MAX_RUNTIME_SECONDS} s")
print(f"Naive demo stop : {NAIVE_RUNTIME_SECONDS} s")

## 6. Naive baseline (reference to beat)

Uses the pre-built `satellites` for propagation and `catalog` only if you want metadata.
Internally samples a time grid (example step = 2 minutes) and checks pairs until the demo time limit.

Intentionally slow — that is the point of the hackathon.

In [ ]:
def naive_baseline_find_close_approaches(
    catalog: Catalog,
    satellites: Dict[int, EarthSatellite],
    propagate_from_utc: datetime,
    propagate_until_utc: datetime,
    close_approach_threshold_km: float,
    max_runtime_seconds: float,
) -> List[ConjunctionClaim]:
    """Naive O(N²) baseline.

    Time step is defined inside this function (example only).
    Uses the pre-built satellites dict — does not call catalog_to_satellites.
    """
    # Example time step for THIS baseline only — not part of the student API
    time_step_seconds: float = 2 * 60

    object_ids: List[int] = sorted(satellites.keys())
    if len(object_ids) < 2:
        return []

    sample_skyfield_times = time_grid(
        propagate_from_utc, propagate_until_utc, time_step_seconds,
    )
    sample_times: List[datetime] = datetimes_of(sample_skyfield_times)
    positions_by_id = propagate_many(satellites, sample_skyfield_times)

    claims: List[ConjunctionClaim] = []
    pairs_checked: int = 0
    total_pairs: int = len(object_ids) * (len(object_ids) - 1) // 2
    t_start: float = time.perf_counter()

    for id_a, id_b in combinations(object_ids, 2):
        if time.perf_counter() - t_start >= max_runtime_seconds:
            elapsed: float = time.perf_counter() - t_start
            print(
                f"  Naive baseline stopped at {elapsed:.1f} s "
                f"after {pairs_checked:,} / {total_pairs:,} pairs "
                f"(expected — full O(N²) would take hours)."
            )
            break
        closest_time, closest_distance_km, _ = closest_approach_on_grid(
            positions_by_id[id_a], positions_by_id[id_b], sample_times,
        )
        pairs_checked += 1
        if closest_distance_km <= close_approach_threshold_km:
            claims.append(ConjunctionClaim(
                norad_a=id_a,
                norad_b=id_b,
                tca_utc=closest_time,
                min_distance_km=closest_distance_km,
                algorithm_id="naive_baseline",
            ))

    claims.sort(
        key=lambda c: float("inf") if c.min_distance_km is None else c.min_distance_km
    )
    return claims


print("Running naive baseline…")
t0: float = time.perf_counter()
baseline_raw: List[ConjunctionClaim] = naive_baseline_find_close_approaches(
    catalog,
    satellites,
    propagate_from_utc,
    propagate_until_utc,
    CLOSE_APPROACH_THRESHOLD_KM,
    NAIVE_RUNTIME_SECONDS,
)
elapsed: float = time.perf_counter() - t0
print(f"Done in {elapsed:.2f} s | claims found: {len(baseline_raw)}")
for claim in baseline_raw[:5]:
    dist = "n/a" if claim.min_distance_km is None else f"{claim.min_distance_km:.3f} km"
    print(f"  {claim.norad_a}–{claim.norad_b}: {dist} at {claim.tca_utc.isoformat()}")

## 7. Your solution

Implement `find_close_approaches` in the next cell.

**Arguments**
| Name | Type | Meaning |
|------|------|---------|
| `catalog` | `Catalog` | Orbital data for filtering / screening |
| `satellites` | `Dict[int, EarthSatellite]` | Pre-built propagators (do not rebuild all of them) |
| `propagate_from_utc` / `propagate_until_utc` | `datetime` | Search window |
| `close_approach_threshold_km` | `float` | Distance threshold (10 km) |
| `max_runtime_seconds` | `float` | Stop and return what you have |

**Return** `List[ConjunctionClaim]` with at least `norad_a`, `norad_b`, `tca_utc`.  
Optional: `min_distance_km`, `algorithm_id`.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# YOUR SOLUTION — implement the function below
# ══════════════════════════════════════════════════════════════════

def find_close_approaches(
    catalog: Catalog,
    satellites: Dict[int, EarthSatellite],
    propagate_from_utc: datetime,
    propagate_until_utc: datetime,
    close_approach_threshold_km: float,
    max_runtime_seconds: float,
) -> List[ConjunctionClaim]:
    """Find pairs that come within close_approach_threshold_km.

    Use `catalog` for orbital fields (e.g. inclination).
    Use `satellites` for propagation — already built outside the timer.
    """
    raise NotImplementedError("Implement your algorithm here.")

## 8. Verifier

For each claim the verifier:

1. Propagates the two satellites at the **claimed time** (`tca_utc`)
2. Measures distance at that exact time
3. Marks **True** if distance ≤ threshold, otherwise **Failed**

It also records how long your function ran. If runtime exceeds the 1-minute limit, claims are still evaluated — the summary just notes that the limit was exceeded.

In [ ]:
@dataclass
class EvaluationReport:
    algorithm_name: str
    close_approaches_detected: int
    claims_verified_ok: int
    claims_failed: int
    runtime_seconds: float
    runtime_limit_seconds: float
    runtime_exceeded: bool
    messages: List[str] = field(default_factory=list)
    claims: List[ConjunctionClaim] = field(default_factory=list)

    def print_summary(self) -> None:
        print(f"=== {self.algorithm_name} ===")
        print(f"  Close approaches detected : {self.close_approaches_detected}")
        print(f"  Verified True             : {self.claims_verified_ok}")
        print(f"  Failed                    : {self.claims_failed}")
        print(
            f"  Runtime                   : {self.runtime_seconds:.2f} s "
            f"(limit {self.runtime_limit_seconds:.0f} s)"
        )
        if self.runtime_exceeded:
            print(
                f"  ⚠ Runtime exceeded the {self.runtime_limit_seconds:.0f} s limit "
                f"(results are still evaluated)."
            )
        for msg in self.messages:
            print(f"  - {msg}")


def verify_claim_at_reported_time(
    claim: ConjunctionClaim,
    satellites: Dict[int, EarthSatellite],
    close_approach_threshold_km: float,
) -> Tuple[bool, float, str]:
    sat_a = satellites.get(claim.norad_a)
    sat_b = satellites.get(claim.norad_b)
    if sat_a is None or sat_b is None:
        missing: List[str] = []
        if sat_a is None:
            missing.append(str(claim.norad_a))
        if sat_b is None:
            missing.append(str(claim.norad_b))
        return False, float("nan"), f"Missing satellites: {', '.join(missing)}"

    try:
        t = time_grid(claim.tca_utc, claim.tca_utc, step_seconds=1.0)
        times: List[datetime] = datetimes_of(t)
        local_sats: Dict[int, EarthSatellite] = {
            claim.norad_a: sat_a,
            claim.norad_b: sat_b,
        }
        positions = propagate_many(local_sats, t)
        _, distance_km, _ = closest_approach_on_grid(
            positions[claim.norad_a],
            positions[claim.norad_b],
            times,
        )
    except Exception as exc:
        return False, float("nan"), f"Propagation failed: {exc}"

    if distance_km <= close_approach_threshold_km:
        return True, distance_km, "True"
    return False, distance_km, (
        f"Failed: distance at claimed time is {distance_km:.6f} km "
        f"> threshold {close_approach_threshold_km:.6f} km"
    )


def evaluate_finder(
    find_fn: Callable[..., List[ConjunctionClaim]],
    catalog: Catalog,
    satellites: Dict[int, EarthSatellite],
    propagate_from_utc: datetime,
    propagate_until_utc: datetime,
    close_approach_threshold_km: float,
    algorithm_name: str,
    max_runtime_seconds: float = 60.0,
) -> EvaluationReport:
    t0: float = time.perf_counter()
    raw_claims: List[ConjunctionClaim] = find_fn(
        catalog,
        satellites,
        propagate_from_utc,
        propagate_until_utc,
        close_approach_threshold_km=close_approach_threshold_km,
        max_runtime_seconds=max_runtime_seconds,
    )
    runtime_seconds: float = time.perf_counter() - t0
    runtime_exceeded: bool = runtime_seconds > max_runtime_seconds

    claims: List[ConjunctionClaim] = list(raw_claims)
    ok_count: int = 0
    fail_count: int = 0
    messages: List[str] = []

    for claim in claims:
        ok, distance_km, message = verify_claim_at_reported_time(
            claim,
            satellites,
            close_approach_threshold_km,
        )
        if ok:
            ok_count += 1
        else:
            fail_count += 1
            messages.append(
                f"FAIL {claim.norad_a}–{claim.norad_b} @ {claim.tca_utc.isoformat()} | {message}"
            )

    return EvaluationReport(
        algorithm_name=algorithm_name,
        close_approaches_detected=len(raw_claims),
        claims_verified_ok=ok_count,
        claims_failed=fail_count,
        runtime_seconds=runtime_seconds,
        runtime_limit_seconds=max_runtime_seconds,
        runtime_exceeded=runtime_exceeded,
        messages=messages,
        claims=claims,
    )


print("Evaluator ready.")

### Run verifier on the naive baseline

In [ ]:
baseline_report: EvaluationReport = evaluate_finder(
    naive_baseline_find_close_approaches,
    catalog,
    satellites,
    propagate_from_utc,
    propagate_until_utc,
    CLOSE_APPROACH_THRESHOLD_KM,
    "naive_baseline",
    max_runtime_seconds=NAIVE_RUNTIME_SECONDS,
)
baseline_report.print_summary()

### Run verifier on your solution

Re-run this cell whenever you change `find_close_approaches` above.

In [ ]:
student_report: EvaluationReport | None
try:
    student_report = evaluate_finder(
        find_close_approaches,
        catalog,
        satellites,
        propagate_from_utc,
        propagate_until_utc,
        CLOSE_APPROACH_THRESHOLD_KM,
        "my_solution",
        max_runtime_seconds=MAX_RUNTIME_SECONDS,
    )
    student_report.print_summary()
except NotImplementedError:
    print("Implement find_close_approaches in the solution cell first.")
    student_report = None

### Side-by-side summary

In [ ]:
if student_report is not None:
    print(
        f"{'algorithm':<20} {'detected':>10} {'true':>8} "
        f"{'failed':>8} {'seconds':>10} {'over_limit':>10}"
    )
    for report in (baseline_report, student_report):
        print(
            f"{report.algorithm_name:<20} "
            f"{report.close_approaches_detected:>10} "
            f"{report.claims_verified_ok:>8} "
            f"{report.claims_failed:>8} "
            f"{report.runtime_seconds:>10.2f} "
            f"{'YES' if report.runtime_exceeded else 'no':>10}"
        )
else:
    print("Your solution did not run yet.")

## 9. Optional: plot one close approach

Plots the closest verified claim (or the best baseline claim if your solution is not ready).

In [ ]:
report_to_plot: EvaluationReport = (
    student_report if student_report is not None else baseline_report
)

if report_to_plot.claims:
    best: ConjunctionClaim = report_to_plot.claims[0]
    zoom = time_grid(
        best.tca_utc - timedelta(minutes=45),
        best.tca_utc + timedelta(minutes=45),
        step_seconds=30.0,
    )
    fig = plot_pair_with_distance(
        satellites[best.norad_a],
        satellites[best.norad_b],
        zoom,
        tca=best.tca_utc,
        norad_a=best.norad_a,
        norad_b=best.norad_b,
        threshold_km=CLOSE_APPROACH_THRESHOLD_KM,
        title=f"Close approach {best.norad_a}–{best.norad_b}",
    )
    save_html(fig, OUTPUT_DIR / "notebook_pair.html")
    fig.show()
else:
    print("No claims to plot.")